In [ ]:
import sys
sys.path.append("../")

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
import io

from mframework.data import ArrayDataset

df = pd.read_parquet("data/mnist_train.parquet")

MEAN = 0.1307
STD  = 0.3081

images = []
for raw in df["image"]:
    img = Image.open(io.BytesIO(raw["bytes"]))
    img = img.convert("L")
    arr = np.array(img, dtype=np.float32) / 255.0  # [0, 1]
    arr = (arr - MEAN) / STD                        # standardise
    arr = arr[np.newaxis, :, :]                     # (1, 28, 28) — channel dim
    images.append(arr)

X = np.stack(images)   # (N, 1, 28, 28)
y = df["label"].to_numpy(dtype=np.int64)

train_ds = ArrayDataset(X, y)

In [ ]:
df = pd.read_parquet("data/mnist_test.parquet")

MEAN = 0.1307
STD  = 0.3081

images = []
for raw in df["image"]:
    img = Image.open(io.BytesIO(raw["bytes"]))
    img = img.convert("L")
    arr = np.array(img, dtype=np.float32) / 255.0
    arr = (arr - MEAN) / STD
    arr = arr[np.newaxis, :, :]         # (1, 28, 28)
    images.append(arr)

X = np.stack(images)                    # (N, 1, 28, 28)
y = df["label"].to_numpy(dtype=np.int64)

test_ds = ArrayDataset(X, y)

In [ ]:
from mframework.autograd.tensor import Tensor
from mframework.optim.sgd import SGD
from mframework.nn import Linear, Sequential, CrossEntropyLoss, ReLU, Conv2D, MaxPool2D, Flatten
from mframework.data import DataLoader, SequentialSampler, BatchSampler
from mframework.dtypes import DType
from tqdm import tqdm
import numpy as np

# Basic feedforward neural network model definition
model = Sequential(
    # 28x28 -> 28x28
    Conv2D(
        in_channels=1, 
        out_channels=6, 
        kernel_size=5,
        padding=2, 
        bias=True
    ),
    ReLU(),
    # 28x28 -> 14x14
    MaxPool2D(
        kernel_size=2,
        stride=2,
        padding=0
    ),
    # 14x14 -> 10x10
    Conv2D(
        in_channels=6, 
        out_channels=16, 
        kernel_size=5,
        padding=0,
        bias=True
    ),
    # 10x10 -> 5x5
    MaxPool2D(
        kernel_size=2,
        stride=2,
        padding=0
    ),
    # 16 * 5 * 5 = 400
    Flatten(),
    Linear(400, 120, True),
    ReLU(),
    Linear(120, 84, True),
    ReLU(),
    Linear(84, 10, True)
)

criterion = CrossEntropyLoss()
lr = 0.01
optim = SGD(model.parameters(), lr)
epochs = 20
seq_sampler = SequentialSampler(train_ds)
batch_sampler = BatchSampler(seq_sampler, batch_size=64, drop_last=False)
train_dl = DataLoader(train_ds, batch_sampler)

for epoch in range(epochs):
    epoch_loss = 0.0
    for i, batch in enumerate(tqdm(train_dl, desc=f"epoch {epoch+1}")):
        # DataLoader yields (X_list, y_list) because of collate implementation
        X_list, y_list = batch

        # Stack / convert to numpy arrays
        X_np = np.stack(X_list).astype(np.float32)
        y_np = np.array(y_list, dtype=np.intp).reshape(-1)

        # Wrap inputs/targets into Tensors
        X_t = Tensor(X_np)
        y_t = Tensor(y_np, dtype=DType.INT64)

        # Run model forward pass and then update params
        logits = model(X_t)
        loss = criterion(logits, y_t)

        epoch_loss += loss.item

        loss.backward()
        optim.step()
        optim.zero_grad()

    avg = epoch_loss / len(train_dl)
    print(f"Epoch {epoch+1}/{epochs} avg loss: {avg:.4f}")


epoch 1: 100%|██████████| 938/938 [03:02<00:00,  5.15it/s]


Epoch 1/20 avg loss: 0.6494


epoch 2: 100%|██████████| 938/938 [02:51<00:00,  5.48it/s]


Epoch 2/20 avg loss: 0.2421


epoch 3: 100%|██████████| 938/938 [02:49<00:00,  5.54it/s]


...

Epoch 10/20 avg loss: 0.0751


epoch 11: 100%|██████████| 938/938 [03:31<00:00,  4.44it/s]


Epoch 11/20 avg loss: 0.0692


epoch 12:  67%|██████▋   | 628/938 [02:15<01:07,  4.62it/s]


KeyboardInterrupt: 

In [ ]:
seq_sampler_test = SequentialSampler(test_ds)
batch_sampler_test = BatchSampler(seq_sampler_test, batch_size=64, drop_last=False)
test_dl = DataLoader(test_ds, batch_sampler_test)

correct = 0
total = 0
test_loss = 0.0

for batch in test_dl:
    X_list, y_list = batch
    # Stack / convert to numpy arrays
    X_np = np.stack(X_list).astype(np.float32)
    y_np = np.array(y_list, dtype=np.intp).reshape(-1)

    # Wrap inputs/targets into Tensors
    X_t = Tensor(X_np)
    y_t = Tensor(y_np, dtype=DType.INT64)
    logits = model(X_t)
    batch_loss = criterion(logits, y_t)
    test_loss += float(batch_loss.data)
    preds = np.argmax(logits.data, axis=1)
    correct += np.sum(preds == y_np)
    total += len(y_np)

avg_loss = test_loss / len(test_dl)
accuracy = correct / total
print(f"Test loss: {avg_loss:.4f}, Test accuracy: {accuracy*100:.2f}%")


Test loss: 0.0408, Test accuracy: 98.69%
